<a href="https://colab.research.google.com/github/dominiksakic/NETworkingMay/blob/main/26_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer weights/brains
- Embedding layers: Words converted into dense vectors.
- Attention Layers: Query, Key and Value Matrices.
- Feedforward layers: like in fully connected NN.
- Layer Norm
- Final output

# Intuition about - attention
- focused part of data, which is the result of an attention filter applied to the original.
- then focus on another part of the data, via another attention filter.
- Each focused result gets concated together and then projected back into the original dimension to avoid increasing the model size.

# What is so special?
- Linear layer: each output depends on each input. (Too much computation for large input and each output)
- Multi - head attention layer: every output depends on every input. But with less computation and weights.

# How does it do it?
- Linear layers: creates each output by taking the total input and different weights for each output. (Weighted sums)

- Attention layers
 - 1. store less amount weights, but can produce the weights in the process.*
 - 2. reduces the computation needed:
  - First a row wise computation and then a column wise computation. (Smilair to a Seperable Convolution)

 * *Dont stores  giants weights directly, they store a smaller amount of weights. The smaller amount of weights get created dynamically with the input to calculate the output/weighted sums.


 # Side notes:
 - Convolution are based on a small spacial area of the input.
 - Convolution share the exact weights with the other outputs.
 - CNNS: Local focus, shared weights across space.

 - Recurrent NN are based on a single point in of the input with a limited aggregation of the past.
 - The weights are shared with every other point in the sequence.
 - RNNS: Process input step-by-step, with shared weights overtime.

 - Transformers, each output still depends on the entire input and it still gets it unique weights.
 - Dynamic weight creation and the decomposition of the computation -> makes it so effective.
 - Full input context at every layer; dynamically computed weights allow flexible, parallel processing.





In [2]:
import numpy as np
"""
1.
 the cat sat

2. Token Vectors
[
  [0.1, 0.3, 0.5],  # embedding of word1
  [0.7, 0.2, 0.9],  # embedding of word2
  [0.0, 0.4, 0.2]   # embedding of word3
]

3. Self attention

scores = [
  dot(word1, word1),
  dot(word1, word2),
  dot(word1, word3)
]

4. Scale and apply
 - helps stabilize gradient
 - turns raw similarity scores into probabilites
 scores = [0.6, 0.3, 0.1]  # sum = 1


6. Sum: Context-aware vector
new_pivot_representation =
  word1 * 0.6 +
  word2 * 0.3 +
  word3 * 0.1

output[i] = new_pivot_representation

7. Repeat for the rest of the words and return this enriched Output sequence!


Another Way to think about it is like this:
1. Column (give us the attention scores)
the (focus) cat sat
the  dot(word1, word1)
cat  ...
sat  ...

2. Then we go into the rows
output[word1] = the*(dot(word1, word1))  + cat * dot(word1, word2) + sat * dot(word1, word1)
"""


# Pseudo Code for self-attention
def self_attention(input_sequence):
  output = np.zeros(shape=input_sequence.shape)
  # iterate over the input sequence
  for i , pivot_vector in enumerate(input_sequence):
    scores = np.zeros(shape=(len(input_sequence),))
    for j, vector in enumerate(input_sequence):
      # Compute attention score/dot product
      scores[j] = np.dot(pivot_vector, vector.T)
    # Normalize and apply a softmax
    scores /= np.sqrt(input_sequence.shape[1])
    scores = softmax(scores)
    new_pivot_representation = np.zeros(shape=pivot_vector.shape)
    for j, vector in enumerate(input_sequence):
      # Take the sum of all tokens weighted by the attention scores.
      new_pivot_representation += vector * scores[j]
    output[i] = new_pivot_representation
  return output
